# Generate amorphous LSU networks

This notebook demonstrates `generate_lsu_network`, which implements the
Wooten-Winer-Weaire simulated annealing algorithm of Sellers et al.
(*Nat. Commun.* **8**, 14439, 2017) on a periodic 3-regular graph.

Documentation: see the `claude_context/` folder.

**Inputs**: target LSU (`lsu_degree_12` or `lsu_degree_22`), one of
(`num_vertices` or `num_rods`), `bounds_microns`.

**Output**: NumPy array `(R, 6)` where each row is `[x1, y1, z1, x2, y2, z2]` —
directly usable in the `create_permittivity_grid_penlike` pipeline. With the
default `pbc_duplicate_boundary_rods=True`, `R` equals the unique-edge count
plus the number of edges crossing box faces (each rendered twice, once from
each canonical-box endpoint), matching the Sellers reference file convention.

In [1]:
import numpy as np
import os
import lsu_network as lsu
import jax 

print('JAX available:', lsu.HAS_JAX)
print("Devices:", jax.devices())

JAX available: True
Devices: [CudaDevice(id=0)]


## Reproduce the example: 1000 vertices / 1500 unique edges, periodicity 11.44 µm

The reference example (`Example/lsu_example_ends.txt`) has Φ_12 ≈ 0.99 and
Φ_22 ≈ 0.89, **N=1000 vertices, E=1500 unique edges**, periodicity 11.44 µm,
mean rod length 0.8 µm. The 1653 lines in that file include 153 PBC-image
duplicates of edges crossing box faces — required so that
`create_permittivity_grid_penlike` (which draws each rod as a literal cylinder
and does not apply PBC) produces a periodic permittivity grid.

At full scale this needs ~50,000 WWW iterations. With JAX it's tractable
(JIT-compiled energy + autodiff gradient); without JAX, plan to run
overnight or reduce iterations.

In [ ]:
# Adjust n_www_iterations down if you want a quicker (less converged) run.
# num_vertices=1000 matches the Sellers reference topology (N=1000 / E=1500).
# With pbc_duplicate_boundary_rods=True (default) each face-crossing edge is
# emitted twice, so len(rods) = 1500 + (boundary duplicates) — close to but
# not exactly 1653 (depends on the specific edge geometry).
rods = lsu.generate_lsu_network(
    lsu_degree_22=0.89,            # could also use lsu_degree_12=0.99
    num_vertices=1000,
    bounds_microns=14.3,
    edge_length=1.0,
    n_www_iterations=10_000,
    initial_temperature=0.5,
    final_temperature=1e-5,
    target_tolerance=0.01,
    check_lsu_every=500,
    relax_local_iters=100,
    relax_global_every=1000,
    relax_global_iters=100,
    local_shell_depth=4,           # Vink/MB local-shell relax (Sellers refs [13,14])
    seed=42,
    use_jax=True,
    verbose=True,
    energy_weights={'alpha':80, 'beta':5, 'gamma':1, 'delta':0.5}
)
print('shape:', rods.shape)

## Save the output

The 6-column form (x1,y1,z1,x2,y2,z2) is directly compatible with
`np.loadtxt` as used by the rest of the pipeline. The 7-column form below
(with a 1-based index column) matches `Example/lsu_example_ends.txt`.

In [3]:
os.makedirs('./Example', exist_ok=True)

# 6-column compatible with create_permittivity_grid_penlike
np.savetxt('./Example/lsu_generated_3.txt', rods,
           fmt=' '.join(['%.14g'] * 6), delimiter='\t')

# # 7-column with index, matching Example/lsu_example_ends.txt
# indexed = np.column_stack([np.arange(1, len(rods) + 1), rods])
# np.savetxt('./Example/lsu_generated_indexed.txt', indexed,
#            fmt='%d\t' + '\t'.join(['%.14g'] * 6))

print('saved', rods.shape[0], 'rods')

saved 1653 rods


## Verify the result

Quick checks: connectivity (rods belong to one connected network), edge length
distribution, and final LSU values.

In [4]:
BOX = 14.3  # Must match bounds_microns above for the stats below to be meaningful.
p1 = rods[:, :3]
p2 = rods[:, 3:]
lengths = np.linalg.norm(p2 - p1, axis=1)

# 1) Rod-length distribution. Reference example (1653 rods, BOX=11.44):
#    mean=0.800 std=0.029  q5=0.752 med=0.801 q95=0.846 min=0.667 max=0.884
qs = np.quantile(lengths, [0.0, 0.05, 0.25, 0.5, 0.75, 0.95, 1.0])
print(f'rod count : {len(rods)}')
print(f'lengths   : mean={lengths.mean():.3f} std={lengths.std():.3f}')
print(f'  quartiles  min={qs[0]:.3f}  5%={qs[1]:.3f}  Q1={qs[2]:.3f}  '
      f'med={qs[3]:.3f}  Q3={qs[4]:.3f}  95%={qs[5]:.3f}  max={qs[6]:.3f}')
print(f'  ref target  mean=0.800 std=0.029 (reach with enough WWW iters)')

# 2) Spatial-coverage check — tile the canonical box into 1 µm cells and
# count cells with no vertex. Reference example: 54.5% empty (Poisson at
# density 0.74 verts/µm³ would naturally give ~44% empty). The thing the
# old configuration-model seed got wrong was *clusters* of empty cells —
# multi-µm voids. The Poisson-disk seed used now should give a roughly
# Poisson-like empty-cell pattern with no large connected void region.
half = BOX / 2.0
verts = np.vstack([p1, p2])
verts_canon = verts - BOX * np.round(verts / BOX)
n_cells = int(np.ceil(BOX))
edges_grid = np.linspace(-half, half, n_cells + 1)
H, _ = np.histogramdd(verts_canon, bins=(edges_grid, edges_grid, edges_grid))
empty = int(np.sum(H == 0))
print(f'1 µm³ vertex coverage: {H.size} cells, {empty} empty '
      f'({empty / H.size:.1%})  (reference: 54.5%)')

try:
    from scipy.ndimage import label
    labeled, n_components = label(H == 0)
    sizes = sorted((int((labeled == c).sum()) for c in range(1, n_components + 1)),
                   reverse=True)
    print(f'  largest empty clusters: {sizes[:5]}  '
          f'(big single cluster is normal at this density due to PBC '
          f'percolation; what was *wrong* before was a big cluster on a '
          f'box face)')
except ImportError:
    pass

print(f'box span   : x [{p1[:,0].min():.3f}, {p1[:,0].max():.3f}] '
      f'y [{p1[:,1].min():.3f}, {p1[:,1].max():.3f}] '
      f'z [{p1[:,2].min():.3f}, {p1[:,2].max():.3f}]')

# 3) Voxel-density uniformity — the test that surfaced the void-clustering
# issue. Bin rod midpoints into a 4x4x4 grid (cells of side ~2.86 µm) and
# measure spread + boundary-vs-interior bias. The ~1µm coverage check above
# is too fine to see this — at 1µm there are ~3 midpoints/cell, so empty
# cells dominate either way. The 4³ grid has ~26 midpoints/expected and
# resolves cluster/void blocks of ~1.4 µm scale.
#
# Reference example (lsu_example_ends.txt):
#    4³ voxels:  std=2.79  min=17  max=30   bdry mean=23.50   int mean=22.88
# Pre-fix run (alpha=10, beta=gamma=delta=1, relax_global_every=500):
#    4³ voxels:  std=7.31  min=8   max=44   bdry mean=24.11   int mean=27.38
# Pass criteria for the option-A fix: std <= 3.5 and min >= 14.
midpts = (p1 + p2) / 2.0
midpts_canon = midpts - BOX * np.round(midpts / BOX)
vbins = np.linspace(-half, half, 5)
Hv, _ = np.histogramdd(midpts_canon, bins=(vbins, vbins, vbins))
bdry_mask = np.zeros(Hv.shape, dtype=bool)
bdry_mask[0, :, :] = bdry_mask[-1, :, :] = True
bdry_mask[:, 0, :] = bdry_mask[:, -1, :] = True
bdry_mask[:, :, 0] = bdry_mask[:, :, -1] = True
expected = midpts_canon.shape[0] / Hv.size
print(f'4³ voxel midpoints (cell ~{BOX/4:.2f} µm, expected {expected:.1f}/cell):')
print(f'  spread     mean={Hv.mean():.2f} std={Hv.std():.2f} '
      f'min={Hv.min():.0f} max={Hv.max():.0f}')
print(f'  bdry/int   bdry mean={Hv[bdry_mask].mean():.2f}  '
      f'int mean={Hv[~bdry_mask].mean():.2f}  '
      f'(ref: 23.50 / 22.88; pre-fix: 24.11 / 27.38)')
print(f'  ref target std=2.79 min=17 max=30  (pass: std<=3.5, min>=14)')


rod count : 1653
lengths   : mean=1.008 std=0.020
  quartiles  min=0.948  5%=0.977  Q1=0.994  med=1.007  Q3=1.020  95%=1.041  max=1.085
  ref target  mean=0.800 std=0.029 (reach with enough WWW iters)
1 µm³ vertex coverage: 3375 cells, 2373 empty (70.3%)  (reference: 54.5%)
  largest empty clusters: [2360, 3, 2, 1, 1]  (big single cluster is normal at this density due to PBC percolation; what was *wrong* before was a big cluster on a box face)
box span   : x [-7.140, 7.126] y [-7.135, 7.136] z [-7.146, 7.146]
4³ voxel midpoints (cell ~3.58 µm, expected 25.8/cell):
  spread     mean=25.83 std=9.36 min=8 max=57
  bdry/int   bdry mean=25.23  int mean=30.00  (ref: 23.50 / 22.88; pre-fix: 24.11 / 27.38)
  ref target std=2.79 min=17 max=30  (pass: std<=3.5, min>=14)
